In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import joblib

In [10]:
train = pd.read_csv("/Users/syedalihussain/Documents/customer_churn/data/processed/train.csv")
test = pd.read_csv("/Users/syedalihussain/Documents/customer_churn/data/processed/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (5634, 20)
Test shape: (1409, 20)


In [11]:
X_train = train.drop(columns=['Churn'])
y_train = train['Churn']

X_test = test.drop(columns=['Churn'])
y_test = test['Churn']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (5634, 19)
X_test shape: (1409, 19)


In [12]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [13]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

print('Logistic Regression trained')

Logistic Regression trained


In [14]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print('Random Forest trained')

Random Forest trained


In [15]:
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

print("XGBoost trained")

XGBoost trained


In [16]:
# Store all models in a dictionary to loop through easily
models = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

# Evaluate each model on test data
print(f"{'Model':<25} {'Accuracy':<12} {'F1 Score':<12} {'ROC-AUC':<12}")
print("-" * 60)

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    print(f"{name:<25} {accuracy:<12.4f} {f1:<12.4f} {roc_auc:<12.4f}")


Model                     Accuracy     F1 Score     ROC-AUC     
------------------------------------------------------------
Logistic Regression       0.8006       0.5933       0.8403      
Random Forest             0.7913       0.5586       0.8222      
XGBoost                   0.7786       0.5530       0.8185      


In [16]:
print(type(X_train), type(X_test))
print(X_train.shape, X_test.shape)
print("Train columns:", getattr(X_train, 'columns', 'No columns'))
print("Test columns:", getattr(X_test, 'columns', 'No columns'))

<class 'pandas.DataFrame'> <class 'pandas.DataFrame'>
(5634, 19) (5635, 0)
Train columns: Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='str')
Test columns: Index([], dtype='str')


In [9]:
X_test.shape

(5635, 0)

In [17]:
import os

os.makedirs("/Users/syedalihussain/Documents/customer_churn/models/saved_models", exist_ok=True)

joblib.dump(xgb_model, "/Users/syedalihussain/Documents/customer_churn/models/saved_models/xgb_model.pkl")

print('Best model saved')

Best model saved


In [18]:
path = "/Users/syedalihussain/Documents/customer_churn/models/saved_models/"
print("Files in saved_models:", os.listdir(path))

Files in saved_models: ['xgb_model.pkl']
